In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from os.path import join as pjoin
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr, zscore, kendalltau
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/reward_distance'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
pos_distances = np.arange(0, (reward_bin_size*4) + reward_bin_size, reward_bin_size) ## distance from reward locations
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 1 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Get the distance of each cell's peak activity is from the reward locations.

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Find the peak of each place field
pf_peaks = np.max(tuning_curves, axis=1)
## Find the spatial bins where each peak occurred
field_dist = np.zeros(pf_peaks.shape[0])
for idx, peak in enumerate(pf_peaks):
    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

pos_vals = []
for start in pos_distances:
    rw_one_amount = np.sum((abs(field_dist - first_rw_pos) >= start) & (abs(field_dist - first_rw_pos) < start + reward_bin_size))
    rw_two_amount = np.sum((abs(field_dist - second_rw_pos) >= start) & (abs(field_dist - second_rw_pos) < start + reward_bin_size))
    pos_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

all_vals = []
for pos in all_midpoints:
    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))
    all_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

In [ ]:
## Example proportion of place fields for all distances centered at x-axis values
fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
fig.add_trace(go.Scattergl(x=all_midpoints * conversion, y=all_vals, mode='lines+markers', marker_size=8))
fig.update_yaxes(range=[0, np.max(all_vals) + 0.01])
fig.show()

In [ ]:
## Example proportion of place fields for only positive differences
fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
fig.add_trace(go.Scattergl(x=pos_distances * conversion, y=pos_vals))
fig.update_yaxes(range=[0, 0.16])
fig.show()

In [ ]:
## Example distribution of place fields across track
rw_bins = np.arange(0, 6.28 + reward_bin_size, reward_bin_size)
H, xbin = np.histogram(field_dist, bins=rw_bins)
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Proportion Place Fields', width=600)
hnorm = H / np.sum(H) ## convert to proportion
fig.add_trace(go.Bar(x=xbin, y=hnorm, marker_color=ce_colors_dict['Multi-context'], marker_line_width=2, marker_line_color='black'))
for val in [reward_one_pos, reward_two_pos]:
    fig.add_vline(x=val, line_dash='dash', line_width=3, opacity=0.7, line_color='red')
fig.update_yaxes(range=[0, 0.05])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_distribution_of_place_fields_running{only_running}_cordir{correct_dir}.png'), width=500, height=500)

### Plot trial by trial activity for cells around the reward location for the example mouse above.

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

minian_path = pjoin(dpath, f'{experiment}/minian_results/{mouse}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Find the peak of each place field
pf_peaks = np.max(tuning_curves, axis=1)
## Find the spatial bins where each peak occurred
field_dist = np.zeros(pf_peaks.shape[0])
for idx, peak in enumerate(pf_peaks):
    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

## Find distance from reward locations
rw_one_dist = field_dist - first_rw_pos
rw_two_dist = field_dist - second_rw_pos
bin_start, bin_end = 0 - (reward_bin_size / 2), 0 + (reward_bin_size / 2)
rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)
sub_uids = neural_data['unit_id'][active_cells]
sub_uids = sub_uids[(rw1_bool) | (rw2_bool)]

sub_data = sdata.sel(unit_id=sub_uids.values)
trial_raster, _ = pc.trial_raster(sub_data, bin_size=bin_size, binarized=True, correct_dir=correct_dir, only_running=only_running)

## Load A and max projection for the desired mouse
max_proj = ctn.open_minian(pjoin(minian_path, f'{sdata.attrs['date']}/{sdata.attrs['timestamp']}/minian'))['max_proj']
A = ctn.open_minian(pjoin(minian_path, f'{sdata.attrs['date']}/{sdata.attrs['timestamp']}/minian'))['A']
subA = A.sel(unit_id=sub_uids.values)

In [ ]:
## Trial raster for a single neuron
neuron = 15
norm_data = trial_raster[:, :, neuron] / np.max(trial_raster[:, :, neuron])
fig = pf.custom_graph_template(x_title='Position (rad)', y_title='Trial')
fig.add_trace(go.Heatmap(x=bins,
                         y=np.arange(1, trial_raster.shape[0] + 1),
                         z=norm_data,
                         colorscale='viridis'))
for val in [first_rw_pos, second_rw_pos]:
    fig.add_vline(x=val, line_width=3, line_dash='dash', line_color='darkgrey', opacity=0.5)
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(dtick=1)
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{neuron}_{day_of_int}_trial_raster.png'), width=500, height=500)

In [ ]:
## Multiple neuron's trial rasters as different rows
cmap = 'viridis'
fig = pf.custom_graph_template(x_title='', y_title='Trial', rows=4, columns=1, height=800, 
                               vertical_spacing=0.02, master_axes=True, font_size=22)
neuron_list = [0, 5, 8 ,3]
for row, n in enumerate(neuron_list): ## 3, 15, 0, 10, 20, 5, 17 for mc51, 0, 5, 8, 3 for mc44
    norm_data = trial_raster[:, :, n] / np.max(trial_raster[:, :, n])
    fig.add_trace(go.Heatmap(x=bins,
                             y=np.arange(1, trial_raster.shape[0] + 1),
                             z=norm_data,
                             colorscale=cmap,
                             coloraxis='coloraxis1'), row=row + 1, col=1)
    if row < 3:
        fig.update_xaxes(visible=False, row=row + 1, col=1)
for val in [first_rw_pos, second_rw_pos]:
    fig.add_vline(x=val, line_width=2, line_dash='dash', line_color='darkgrey', opacity=1)
fig.update_yaxes(autorange='reversed', dtick=10)
fig.update_xaxes(title='Position (rad)', dtick=1)
fig.update_layout(coloraxis1=dict(colorscale=cmap))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_example_neuron_rasters.png'), width=500, height=800)

In [ ]:
## Multiple neuron's trial rasters as different rows
cmap = 'gray_r'
fig = pf.custom_graph_template(x_title='', y_title='Trial', rows=4, columns=1, height=800, 
                               vertical_spacing=0.02, master_axes=True, font_size=22)
neuron_list = [8, 5 ,3]
for row, n in enumerate(neuron_list): ## 17, 15, 10 for mc51, 5, 8, 3 for mc44
    norm_data = (trial_raster[:, :, n] > 0).astype(int)
    fig.add_trace(go.Heatmap(x=bins,
                             y=np.arange(1, trial_raster.shape[0] + 1),
                             z=norm_data,
                             colorscale=cmap,
                             coloraxis='coloraxis1',
                             showscale=False), row=row + 1, col=1)
    if row < 3:
        fig.update_xaxes(showticklabels=False, row=row + 1, col=1)
for val in [first_rw_pos, second_rw_pos]:
    fig.add_vline(x=val, line_width=3, line_dash='dash', line_color='red', opacity=1)
fig.update_yaxes(autorange='reversed', dtick=10)
fig.update_xaxes(title='Position (rad)', dtick=1, row=4)
fig.update_layout(coloraxis1=dict(colorscale=cmap, showscale=False))
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_bw_example_neuron_rasters.png'), width=500, height=800)

In [ ]:
## Plot multiple neurons with tuning curve above raster
## Multiple neuron's trial rasters as different rows
cmap = 'gray_r'
fig = pf.custom_graph_template(x_title='', y_title='', rows=6, columns=1, height=700, 
                               vertical_spacing=0.02, row_heights=[0.09, 0.25, 0.09, 0.25, 0.09, 0.25], master_axes=False, font_size=22)
neuron_list = [17, 15, 10]
for row, n in enumerate(neuron_list):
    if row == 0:
        r = 1
    elif row == 1:
        r = 3
    elif row == 2:
        r = 5
    avg_fr = np.mean(trial_raster[:, :, n], axis=0) / np.max(np.mean(trial_raster[:, :, n], axis=0)) ## normalized
    fig.add_trace(go.Scattergl(x=bins,
                               y=avg_fr,
                               mode='lines', line_color='black',
                               line_width=1.5, showlegend=False), row=r, col=1)

for row, n in enumerate(neuron_list): ## 17, 15, 10 for mc51, 5, 8, 3 for mc44
    if row == 0:
        r = 2
    elif row == 1:
        r = 4 
    elif row == 2:
        r = 6
    norm_data = (trial_raster[:, :, n] > 0).astype(int)
    fig.add_trace(go.Heatmap(x=bins,
                             y=np.arange(1, trial_raster.shape[0] + 1),
                             z=norm_data,
                             colorscale=cmap,
                             coloraxis='coloraxis1',
                             showscale=False), row=r, col=1)

for rowval in np.arange(1, 6):
    fig.update_xaxes(showticklabels=False, row=rowval, col=1)
for val in [first_rw_pos, second_rw_pos]:
    for rowval in [2, 4, 6]:
        fig.add_vline(x=val, line_width=3, line_dash='dash', line_color='red', opacity=1, row=rowval)
for rowval in [2, 4, 6]:
    fig.update_yaxes(autorange='reversed', dtick=10, row=rowval)
    fig.update_yaxes(title='Trial', row=rowval)
fig.update_xaxes(title='Position (rad)', dtick=1, row=6)
fig.update_layout(coloraxis1=dict(colorscale=cmap, showscale=False))
for rowval in [1, 3, 5]:
    fig.update_yaxes(title='FR', row=rowval)
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_example_neuron_rasters_frs.svg'), width=500, height=800)

In [ ]:
## Plot spatial footprints on the max projection for each cell above
ylim = (70, 515)
xlim = (70, 515)
norm_max_proj = max_proj.values / np.max(max_proj.values)
for n in neuron_list:
    avals = subA[n, :, :].values
    avals = (avals > 0).astype(float)
    avals[avals == 0] = np.nan
    xaxis = np.arange(xlim[0], xlim[1])
    yaxis = np.arange(xlim[0], ylim[1])
    fig = pf.custom_graph_template(x_title='', y_title='')
    fig.add_trace(go.Heatmap(x=xaxis, y=yaxis, z=norm_max_proj[xlim[0]:xlim[1], ylim[0]:ylim[1]], colorscale='viridis', showscale=False))
    fig.add_trace(go.Heatmap(x=xaxis, y=yaxis, z=avals[xlim[0]:xlim[1], ylim[0]:ylim[1]], colorscale='reds', showscale=False))
    fig.update_yaxes(visible=False)
    fig.update_xaxes(visible=False)
    fig.show(config={'scrollZoom': True})
    fig.write_image(pjoin(fig_path, f'{mouse}_{n}_{day_of_int}_example_spatial_footprint.png'), width=500, height=500)

### Combine across mice to get proportion of place fields around reward locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion': [],
               'lick_accuracy': [], 'rewards': [], 'hr': [], 'cr': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        behav_path = pjoin(dpath, f'{experiment}/output/behav/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            behav_mouse_path = pjoin(behav_path, f'{mouse}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Behavior
                behav = pd.read_feather(pjoin(behav_mouse_path, f'{mouse}_{session.split('_')[-1].split('.')[0]}.feat'))
                behav = behav[~behav['probe']] ## exclude probe
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]    
                lick_acc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                signal = pd.DataFrame(ctb.dprime_metrics(behav, mouse, day=index+1, reward_ports=[reward_one, reward_two], forward_reverse='all'))
                
                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
                    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['proportion'].append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])
                    output_dict['lick_accuracy'].append(lick_acc)
                    output_dict['rewards'].append(np.sum(behav['water']))
                    output_dict['hr'].append(signal.groupby(['day'], as_index=False).agg({'hits': 'mean'})['hits'].values[0])
                    output_dict['cr'].append(signal.groupby(['day'], as_index=False).agg({'CR': 'mean'})['CR'].values[0])
rel_df = pd.DataFrame(output_dict)
rel_df.to_feather(pjoin(int_data, f'max_pf_relative_rewards_{cell_type}_second_def.feat'))

In [ ]:
## Plot proportion of place fields for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
plot_mice = False
avg_rel = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='Proportion Place Fields', width=600, titles=[f'Day {day}'])
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['proportion']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['proportion']['sem'], thickness=2.5)))
if plot_mice:
    for mouse in rel_df['mouse'].unique():
        mdata = rel_df[(rel_df['day'] == day) & (rel_df['mouse'] == mouse)]
        fig.add_trace(go.Scattergl(x=mdata['relative_bin'] * conversion, y=mdata['proportion'], mode='lines', name=mouse,
                                   line_color=ce_colors_dict[mdata['group'].unique()[0]], line_width=1.5, opacity=0.4, showlegend=False))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_proportion_pfs.svg'), width=600, height=500)

In [ ]:
## Look at development of place field representation in A for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
cont = avg[(avg['day'] >= 1) & (avg['day'] < 6)] ## subset for days just in context A
for day in cont['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = cont[(cont['day'] == day) & (cont['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], showlegend=False, mode='lines+markers',
                            legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2, marker_size=9, opacity=1,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'], thickness=2.5)), row=1, col=day)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=-0.5,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Port Distance (cm)':
        fig['layout']['annotations'][idx]['yshift'] = -60
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_contA_multi_two_over_rep.png'), width=1400, height=500)

In [ ]:
## Plot the first day in each new context for Multi-context mice
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields', titles=['Multi-context'])
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    d_data = avg[(avg['day'] == day) & (avg['group'] == 'Multi-context')]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], mode='lines+markers',
                            name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=9,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'], thickness=2.5)))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_context_switches_mc.png'), width=500, height=500)

In [ ]:
## Plot the first day in each new context for Two-context and Multi-context
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='', titles=['Two-context', 'Multi-context'], rows=1, columns=2,
                               shared_x=True, shared_y=True, width=900)
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    for idx, group in enumerate(['Two-context', 'Multi-context']):
        d_data = avg[(avg['day'] == day) & (avg['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], mode='lines+markers', showlegend=False,
                                legendgroup=f'Day {day}', name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=9,
                                marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'], thickness=2.5)), row=1, col=idx + 1)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[0, 0.11], dtick=0.02)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.data[4]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_context_switches_tc_mc.png'), width=1000, height=500)

In [ ]:
## Correlate rewards with the proportion of place fields around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Rewards', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['rewards'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['rewards'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=25)
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_rewards_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['rewards'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['rewards'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

In [ ]:
## Correlate lick accuracy with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Lick Accuracy (%)', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['lick_accuracy'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['lick_accuracy'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=25, range=[0, 100])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_lick_accuracy_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['lick_accuracy'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['lick_accuracy'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

In [ ]:
## Correlate hit rate with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Hit Rate', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['hr'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['hr'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=0.25, range=[0, 1])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_hr_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['hr'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['hr'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

In [ ]:
## Correlate correct rejection rate with the proportion of place fiels around the reward location
day = 16
yvar = 'proportion'
ufunc = pearsonr
sub_df = rel_df[rel_df['day'] == day]
sub_df = sub_df[sub_df['relative_bin'] == 0]

fig = pf.custom_graph_template(x_title='Correct Rejection Rate', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = sub_df[sub_df['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['cr'], y=gdata[yvar], mode='markers', marker_color=ce_colors_dict[group],
                               marker=dict(line=dict(width=1.5, color='black')), marker_size=9, name=group))
    
    ## Linear regression to add line of best fit
    X = gdata['cr'].values.reshape(-1, 1)
    y = gdata[yvar].values.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    x_true = np.linspace(np.min(X), np.max(X), 15).reshape(-1, 1)
    ypred = reg.predict(x_true)
    fig.add_trace(go.Scattergl(x=x_true.flatten(), y=ypred.flatten(), mode='lines', line_color=ce_colors_dict[group],
                               name=group, legendgroup=group, showlegend=False, line_width=2.5))
fig.update_xaxes(dtick=0.25, range=[0, 1])
fig.update_yaxes(range=[-0.01, 0.125])
if cell_type == 'place_cells':
    fig.update_yaxes(range=[-0.01, 0.13], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_corr_cr_reward_overrep_day{day}.png'))
print(f'Two-context: {ufunc(sub_df['cr'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Two-context') & (sub_df['day'] == day)])}')
print(f'Multi-context: {ufunc(sub_df['cr'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)], sub_df['proportion'][(sub_df['group'] == 'Multi-context') & (sub_df['day'] == day)])}')

### Get the proportion of place fields around undershooting, overshooting, or leftover port locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
port_type = 'overshooting'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)
                    
                ## Get positions of the under or over-shooting ports and use those as the zero location
                reward_ports = [sdata.attrs['reward_one'], sdata.attrs['reward_two']]
                front_ports, back_ports = ctb.front_back_ports(reward_list=reward_ports) ## under and over-shooting ports
                if port_type == 'overshooting':
                    first_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == back_ports[0]].values)
                    second_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == back_ports[1]].values)
                elif port_type == 'undershooting':
                    first_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == front_ports[0]].values)
                    second_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == front_ports[1]].values)
                elif port_type == 'leftover':
                    leftover_ports = []
                    for port in np.arange(1, 9):
                        if port in reward_ports + front_ports + back_ports:
                            pass
                        else:
                            leftover_ports.append(port)
                    first_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == leftover_ports[0]].values)
                    second_rw_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == leftover_ports[1]].values)


                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
                    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['proportion'].append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])
rel_df = pd.DataFrame(output_dict)
rel_df.to_feather(pjoin(int_data, f'max_pf_relative_{port_type}_ports_{cell_type}.feat'))

In [ ]:
## Plot proportion of place fields for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='Proportion Place Fields', width=500)
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['proportion']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['proportion']['sem'])))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[0, 0.08], dtick=0.02)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_{port_type}_mc_tc_proportion_pfs.png'), width=500, height=500)

In [ ]:
## Look at development of place field representation in A for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
cont = avg[(avg['day'] >= 1) & (avg['day'] < 6)] ## subset for days just in context A
for day in cont['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = cont[(cont['day'] == day) & (cont['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['proportion']['mean'], showlegend=False, mode='lines+markers',
                            legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2, marker_size=9,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['proportion']['sem'], thickness=2.5)), row=1, col=day)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=-0.5,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Port Distance (cm)':
        fig['layout']['annotations'][idx]['yshift'] = -60
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_contA_mc_tc_over_rep_{port_type}.png'), width=1400, height=500)

### Get the average amount of spatial information of cells x distance from the rewarded locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 
               'avg_spatial_info_rw1': [], 'avg_spatial_info_rw2': [], 'avg_spatial_info': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['avg_spatial_info_rw1'].append(np.mean(sdata['skaggs_info'][rw1_bool].values))
                    output_dict['avg_spatial_info_rw2'].append(np.mean(sdata['skaggs_info'][rw2_bool].values))
                    output_dict['avg_spatial_info'].append(np.mean(np.concatenate((sdata['skaggs_info'][rw1_bool].values, sdata['skaggs_info'][rw2_bool].values))))
si_df = pd.DataFrame(output_dict)
si_df.to_feather(pjoin(int_data, f'spatial_info_relative_rewards_{cell_type}.feat'))

In [ ]:
## Plot the average spatial information for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = si_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='Spatial Info (bits/event)', width=600, titles=[f'Day {day}'])
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['avg_spatial_info']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['avg_spatial_info']['sem'], thickness=2.5)))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[2.5, 9.0], dtick=1.5)
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_avg_spatial_info_pfs.svg'), width=600, height=500)

In [ ]:
## Plot the first day in each new context for Multi-context mice
avg = si_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='Spatial Info (bits/event)', titles=['Multi-context'])
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    d_data = avg[(avg['day'] == day) & (avg['group'] == 'Multi-context')]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['avg_spatial_info']['mean'], mode='lines+markers',
                            name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=7,
                            marker_line_color='black', error_y=dict(type='data', array=d_data['avg_spatial_info']['sem'], thickness=2.5)))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[2.2, 6.6])
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_si_context_switches_mc.png'), width=500, height=500)

In [ ]:
## Plot the first day in each new context for Two-context and Multi-context
avg = si_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='', titles=['Two-context', 'Multi-context'], rows=1, columns=2,
                               shared_x=True, shared_y=True, width=900)
for day in [6, 11, 16]:
    if day == 6:
        c = 'B'
    elif day == 11:
        c = 'C'
    elif day == 16:
        c = 'D'
    for idx, group in enumerate(['Two-context', 'Multi-context']):
        d_data = avg[(avg['day'] == day) & (avg['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'] * conversion, y=d_data['avg_spatial_info']['mean'], mode='lines+markers', showlegend=False,
                                legendgroup=f'Day {day}', name=f'Day {day}', marker_color=context_colors[c], marker_line_width=2, marker_size=7,
                                marker_line_color='black', error_y=dict(type='data', array=d_data['avg_spatial_info']['sem'], thickness=2.5)), row=1, col=idx + 1)
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[2.2, 6.6])
fig.update_yaxes(title='Spatial Info (bits/event)', col=1)
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.data[4]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_si_context_switches_tc_mc.png'), width=1000, height=500)

### Get spatial information around under/overshooting reward locations.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
port_type = 'overshooting'

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'avg_spatial_info': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get where the peak of the tuning curve is for each cell
                field_dist = pc.place_field_peak(tuning_curves, bins)
                    
                ## Get positions of the under or over-shooting ports and use those as the zero location
                front_ports, back_ports = ctb.front_back_ports(reward_list=[sdata.attrs['reward_one'], sdata.attrs['reward_two']]) ## under and over-shooting ports
                if port_type == 'overshooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == back_ports[1]) & (sdata['correct_dir'])].values)
                elif port_type == 'undershooting':
                    first_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[0]) & (sdata['correct_dir'])].values)
                    second_rw_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == front_ports[1]) & (sdata['correct_dir'])].values)
                
                ## Find distance from port locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['avg_spatial_info'].append(np.mean(np.concatenate((sdata['skaggs_info'][rw1_bool].values, sdata['skaggs_info'][rw2_bool].values))))
si_df = pd.DataFrame(output_dict)
si_df.to_feather(pjoin(int_data, f'si_relative_{port_type}_ports_{cell_type}.feat'))

In [ ]:
## Plot the average spatial information for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = si_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_spatial_info': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Port Distance (cm)', y_title='Spatial Info (bits/event)')
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['avg_spatial_info']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['avg_spatial_info']['sem'])))
fig.add_vrect(x0=-2, x1=2, fillcolor=chance_color, opacity=0.5, layer="below", line_width=0)
fig.update_yaxes(range=[2.2, 6.6])
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_{port_type}_mc_tc_avg_spatial_info_pfs.png'), width=500, height=500)

### Find average trial start of place fields across reward distances.

In [ ]:
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'avg_trial_start': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                sdata = xr.load_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                    
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy, bayesian_decoding=False)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Find the peak of each place field
                pf_peaks = np.max(tuning_curves, axis=1)
                ## Find the spatial bins where each peak occurred
                field_dist = np.zeros(pf_peaks.shape[0])
                for idx, peak in enumerate(pf_peaks):
                    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]
                    
                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos
                
                ## Find distance from reward locations
                rw_one_dist = field_dist - first_rw_pos
                rw_two_dist = field_dist - second_rw_pos

                for pos in all_midpoints:
                    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
                    rw1_bool = (rw_one_dist >= bin_start) & (rw_one_dist < bin_end)
                    rw2_bool = (rw_two_dist >= bin_start) & (rw_two_dist < bin_end)

                    sub_ar = sdata[(rw1_bool) | (rw2_bool)]
                    trial_raster, raster_bins = pc.trial_raster(sub_ar, bin_size=bin_size, correct_dir=correct_dir, only_running=only_running)
                    trials_start_per_neuron = pc.place_field_starting_trials(trial_raster)

                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(pos)
                    output_dict['avg_trial_start'].append(np.round(np.mean(trials_start_per_neuron)))
trial_start_df = pd.DataFrame(output_dict)
trial_start_df.to_feather(pjoin(int_data, f'trial_start_relative_rewards_{cell_type}_second_def.feat'))

In [ ]:
## Plot the average stable place field trial start for a specified day for each reward_bin_size location, where the x-axis value is the center of the bin.
day = 16
avg_rel = trial_start_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'avg_trial_start': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Place Field Trial Start')
for group in ['Two-context', 'Multi-context']:
    gdata = avg_rel[(avg_rel['group'] == group) & (avg_rel['day'] == day)]

    fig.add_trace(go.Scattergl(x=gdata['relative_bin'] * conversion, y=gdata['avg_trial_start']['mean'], mode='lines+markers', line_color=ce_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=2, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['avg_trial_start']['sem'])))
fig.update_yaxes(range=[0, 20]) ## y-axes for day 16
fig.show()
fig.write_image(pjoin(fig_path, f'{cell_type}_{day}_mc_tc_avg_trial_start_pfs.png'), width=500, height=500)